<a href="https://colab.research.google.com/github/Innovatewithapple/convnextv2-intel-scenes/blob/main/CNNIntelClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

puneet6060_intel_image_classification_path = kagglehub.dataset_download('puneet6060/intel-image-classification')
mihirvyas_testingimages_path = kagglehub.dataset_download('mihirvyas/testingimages')

print('Data source import complete.')


In [ ]:
import numpy as np
import random
import os
import torch

def setProductionSeed(isActive,seed=42):
  if isActive == True:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
      torch.cuda.manual_seed(seed)
      torch.cuda.manual_seed_all(seed)

      torch.backends.cudnn.deterministic = True
      torch.backends.cudnn.benchmark = False

setProductionSeed(True)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms,datasets
from torch.utils.data import DataLoader
import os
import warnings
from google.colab import userdata
import timm
import pandas as pd
import numpy as np
from PIL import Image,ImageFile
from tqdm import tqdm
import gc

In [ ]:
# os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
# os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [ ]:
# !kaggle datasets download -d puneet6060/intel-image-classification

In [ ]:
# !unzip -q intel-image-classification.zip -d ./data_folder/

In [ ]:
train_path = '/kaggle/input/datasets/puneet6060/intel-image-classification/seg_train/seg_train'
validation_path = '/kaggle/input/datasets/puneet6060/intel-image-classification/seg_test/seg_test'

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(),
    transforms.ToTensor()
])
val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [ ]:
# 1. This is the secret: Turn the Warning into a Crash so 'except' catches it
warnings.filterwarnings("error", category=UserWarning)
count = 0
for root, dirs, files in os.walk(train_path):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            file_path = os.path.join(root, file)
            try:
                img = Image.open(file_path)
                img.load()  # <--- This is the "Truth" test. It forces a full read.
            except Exception as e:
                print(f"❌ Deleting incomplete/corrupt image: {file_path}")
                os.remove(file_path)
                count += 1
# 2. IMPORTANT: Turn warnings back to normal after cleaning
warnings.resetwarnings()
print(f"🧹 Cleanup finished. Total deleted: {count}")

🧹 Cleanup finished. Total deleted: 0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
train_data = datasets.ImageFolder(root=train_path,transform=train_transform)
val_data = datasets.ImageFolder(root=validation_path,transform=val_transform)

In [ ]:
train_loader = DataLoader(dataset=train_data,batch_size=32,shuffle=True,pin_memory=True,num_workers=0)
val_loader = DataLoader(dataset=val_data,batch_size=32,shuffle=False,pin_memory=True,num_workers=0)

In [ ]:
convNxtModel = timm.create_model('convnextv2_tiny.fcmae_ft_in22k_in1k',pretrained=True,num_classes=0)

In [ ]:
custom_head = nn.Sequential(
    nn.LazyLinear(128),
    nn.GELU(),
    nn.Dropout(0.3),
    nn.Linear(128,6)
)

In [ ]:
for name, module in convNxtModel.named_children():
    print(name)

stem
stages
norm_pre
head


In [ ]:
print(len(convNxtModel.stages))

4


In [ ]:
for params in convNxtModel.parameters():
  params.requires_grad = False

for subfolder in list(convNxtModel.stages)[-2:]:
  for params in subfolder.parameters():
    params.requires_grad = True

for params in custom_head.parameters():
  params.requires_grad = True

In [ ]:
final_Model = nn.Sequential(
    convNxtModel,
    custom_head
).to(device)

In [ ]:
optimizer = optim.AdamW(params=filter(lambda p: p.requires_grad,final_Model.parameters()),lr=1e-5,weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

In [ ]:
epochs = 5
best_val_accuracy = 0
for epoch in range(epochs):
  # ---------------- TRAIN ----------------
  final_Model.train()
  train_loss = 0
  train_correct = 0
  train_total = 0

  for images,labels in train_loader:
    images,labels = images.to(device),labels.to(device)

    optimizer.zero_grad()

    with torch.amp.autocast('cuda'):
      model_output = final_Model(images)
      loss = loss_fn(model_output,labels)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    train_loss += loss.item()
    pred = torch.argmax(model_output,dim=1)

    train_correct += (pred == labels).sum().item()
    train_total += labels.size(0)

  train_accuracy = train_correct / train_total
  average_loss = train_loss / len(train_loader)

  # ---------------- VALIDATION ----------------
  with torch.no_grad():
      final_Model.eval()
      val_loss = 0
      val_correct = 0
      val_total = 0

      for images,labels in val_loader:
        images,labels = images.to(device),labels.to(device)

        model_output = final_Model(images)
        loss = loss_fn(model_output,labels)

        val_loss += loss.item()
        pred = torch.argmax(model_output,dim=1)

        val_correct += (pred == labels).sum().item()
        val_total += labels.size(0)

      val_accuracy = val_correct / val_total
      val_average_loss = val_loss / len(val_loader)
      print(f'\nEpochs: {epoch+1}/{epochs}...')
      print(f'\ntrain_acc: {train_accuracy} | train_loss: {average_loss}')
      print(f'\nval_acc: {val_accuracy} | val_loss: {val_average_loss}')

      # ---------------- SAVE BEST MODEL ----------------

  if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        torch.save({
            'epoch': epoch,
            'model_state_dict': final_Model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_accuracy': val_accuracy,
            'config': {
                'backbone': 'convnextv2_tiny.fcmae_ft_in22k_in1k',
                'hidden_dim': 128,
                'num_classes': 6,
                'image_size': 224
            }
        }, "best_model.pth")
        print("Best model saved!")

  gc.collect()
  torch.cuda.empty_cache()



Epochs: 1/5...

train_acc: 0.857631466438649 | train_loss: 0.539157854193707

val_acc: 0.9356666666666666 | val_loss: 0.20431616622954607
Best model saved!

Epochs: 2/5...

train_acc: 0.9420692603676785 | train_loss: 0.18105394664753

val_acc: 0.945 | val_loss: 0.16970204739296374
Best model saved!

Epochs: 3/5...

train_acc: 0.9526863331908223 | train_loss: 0.14286703181904922

val_acc: 0.9476666666666667 | val_loss: 0.15638966130172002
Best model saved!

Epochs: 4/5...

train_acc: 0.9593130967649993 | train_loss: 0.1259462105148474

val_acc: 0.951 | val_loss: 0.1555789830014506
Best model saved!

Epochs: 5/5...

train_acc: 0.9658686048168733 | train_loss: 0.10601299950653573

val_acc: 0.948 | val_loss: 0.15738404802779885


**Testing Begin.....**

In [ ]:
import requests
from PIL import Image
from io import BytesIO

In [ ]:
image = Image.open("/kaggle/input/datasets/mihirvyas/testingimages/24272.jpg").convert("RGB")

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [ ]:
url = "https://images.unsplash.com/photo-1549598685-0058b114c9d6?q=80&w=627&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D"

response = requests.get(url)

image = Image.open(
    BytesIO(response.content)
).convert("RGB")

In [ ]:


image = transform(image)

image = image.unsqueeze(0).to(device)

final_Model.eval()

with torch.no_grad():

    output = final_Model(image)

    pred = torch.argmax(output, dim=1).item()

idx_to_class = {
    v:k for k,v in train_data.class_to_idx.items()
}
print('ALL CATEGORIES:',idx_to_class)
print(idx_to_class[pred])

ALL CATEGORIES: {0: 'buildings', 1: 'forest', 2: 'glacier', 3: 'mountain', 4: 'sea', 5: 'street'}
glacier
